## Actividad 1 — Sistema Experto Básico Basado en Reglas

### ¿Qué estamos construyendo?

Un sistema experto mínimo para diagnosticar fallos en un motor industrial.
Tiene tres componentes:

- **Datos de entrada:** lecturas de sensores del motor
- **Reglas:** conocimiento del experto codificado como funciones IF-THEN
- **Motor de inferencia:** coordina la evaluación y acumula diagnósticos

### Paso 1 — Datos de entrada

Simulamos tres sensores del motor. En un sistema real estos valores
vendrían de una API o base de datos en tiempo real.

In [1]:
# Paso 1: Variables de entrada (simulan lecturas de sensores)
# Cada variable tiene un umbral crítico que las reglas evaluarán.

temperatura     = 85    # °C   → umbral crítico: > 80
vibracion       = 0.9   # u.n. → umbral crítico: > 0.7
consumo_energia = 1500  # W    → umbral crítico: > 1200

print(f"Temperatura:     {temperatura} °C")
print(f"Vibración:       {vibracion}")
print(f"Consumo energía: {consumo_energia} W")

Temperatura:     85 °C
Vibración:       0.9
Consumo energía: 1500 W


### Paso 2 — Reglas del sistema experto

Cada regla se implementa como una **función independiente** que:
- Recibe la lectura de un sensor como parámetro
- Evalúa una condición IF-THEN
- Retorna el diagnóstico como string si la condición se cumple, o `None` si no

> **¿Por qué `None` y no `False`?**  
> Porque el motor de inferencia usará `if resultado:` para filtrar.
> `None` es falsy en Python, lo que permite que el motor no necesite
> conocer los diagnósticos posibles de antemano — solo verifica si
> la regla activó algo.

In [2]:
# Paso 2: Reglas como funciones individuales
# Cada función representa UNA regla de producción (IF-THEN)

def diagnostico_sobrecalentamiento(temperatura):
    # SI temperatura > 80 → Sobrecalentamiento
    if temperatura > 80:
        print("  [Regla 1] Temperatura alta detectada.")
        return "Sobrecalentamiento"
    return None

def diagnostico_falta_lubricacion(vibracion):
    # SI vibración > 0.7 → Falta de lubricación
    if vibracion > 0.7:
        print("  [Regla 2] Vibración alta detectada.")
        return "Falta de lubricación"
    return None

def diagnostico_carga_excesiva(consumo_energia):
    # SI consumo > 1200 W → Carga excesiva
    if consumo_energia > 1200:
        print("  [Regla 3] Consumo de energía alto detectado.")
        return "Carga excesiva"
    return None

print("✓ Reglas definidas.")

✓ Reglas definidas.


### Paso 3 — Motor de inferencia

El motor es el **coordinador central**: aplica todas las reglas en orden
y acumula los diagnósticos que se activan.

Usamos **forward chaining** (encadenamiento hacia adelante):
partimos de los hechos conocidos (lecturas de sensores) y aplicamos
reglas hasta obtener conclusiones.

> **Nota:** el motor evalúa **todas** las reglas sin detenerse al primer
> diagnóstico. Esto permite detectar múltiples problemas simultáneos,
> que es exactamente lo que necesitamos en diagnóstico industrial.

In [3]:
# Paso 3: Motor de inferencia
# Aplica todas las reglas y acumula los diagnósticos activos.

def motor_inferencia(temperatura, vibracion, consumo_energia):
    print(">>> Iniciando evaluación...\n")
    diagnosticos = []  # Acumulador de resultados

    # Aplicar cada regla — si retorna algo distinto de None, lo guardamos
    resultado = diagnostico_sobrecalentamiento(temperatura)
    if resultado:
        diagnosticos.append(resultado)

    resultado = diagnostico_falta_lubricacion(vibracion)
    if resultado:
        diagnosticos.append(resultado)

    resultado = diagnostico_carga_excesiva(consumo_energia)
    if resultado:
        diagnosticos.append(resultado)

    return diagnosticos

print("✓ Motor de inferencia definido.")

✓ Motor de inferencia definido.


### Paso 4 — Ejecución del sistema experto

Llamamos al motor de inferencia con los valores definidos en el Paso 1.
El motor evaluará las tres reglas y retornará la lista de diagnósticos activos.

In [10]:
# Paso 4: Ejecutar el sistema experto

resultado_final = motor_inferencia(temperatura, vibracion, consumo_energia)

print("\n" + "="*50)
print("Diagnósticos emitidos:", resultado_final)
print("="*50)


>>> Iniciando evaluación...

  [Regla 1] Temperatura alta detectada.
  [Regla 2] Vibración alta detectada.
  [Regla 3] Consumo de energía alto detectado.

Diagnósticos emitidos: ['Sobrecalentamiento', 'Falta de lubricación', 'Carga excesiva']


## Actividad 2 — Búsqueda Heurística con el Algoritmo A*

### ¿Para qué sirve A* aquí?

En sistemas expertos complejos, el motor de inferencia puede tener que
explorar muchos caminos posibles para llegar a un diagnóstico.
A* permite encontrar el camino óptimo de forma eficiente usando:
```
f(n) = g(n) + h(n)
        ↑        ↑
  costo real   heurística
  acumulado    (estimación del costo restante)
```

- `g(n)` — lo que ya costó llegar hasta el nodo actual (exacto)
- `h(n)` — estimación de lo que falta hasta el objetivo (aproximado)
- `f(n)` — prioridad del nodo: siempre expandimos el de menor f

### El caso de uso

Una red de estaciones de monitoreo industrial conectadas entre sí.
Queremos encontrar la ruta de menor costo entre dos estaciones.
```
A ──1── B
│       │ \
4       2   5
│       │    \
C ──────╯     D
 \────1───────╯
```

Los números son el tiempo de recorrido entre estaciones.

### Paso 1 — Importar módulo y definir el grafo

Usamos `heapq`, módulo estándar de Python que implementa una
**cola de prioridad** (min-heap).

Es esencial para A* porque necesitamos siempre expandir el nodo
con el **menor f(n)** — y `heapq` garantiza eso en O(log n).

El grafo se define como un diccionario de diccionarios:
- Clave externa: nodo actual
- Clave interna: nodo vecino
- Valor: costo de la arista entre ellos

In [11]:
# Paso 1: Importar módulo y definir el grafo

import heapq

# Grafo de estaciones de monitoreo
# grafo[nodo][vecino] = costo de recorrido
grafo = {
    'A': {'B': 1, 'C': 4},
    'B': {'A': 1, 'C': 2, 'D': 5},
    'C': {'A': 4, 'B': 2, 'D': 1},
    'D': {'B': 5, 'C': 1}
}

print("Conexiones del grafo:")
for nodo, vecinos in grafo.items():
    print(f"  {nodo} → {vecinos}")


Conexiones del grafo:
  A → {'B': 1, 'C': 4}
  B → {'A': 1, 'C': 2, 'D': 5}
  C → {'A': 4, 'B': 2, 'D': 1}
  D → {'B': 5, 'C': 1}


### Paso 2 — Función heurística

La heurística es una **estimación** del costo restante desde cada
nodo hasta el objetivo. Para que A* garantice el camino óptimo,
debe ser **admisible**: nunca puede sobreestimar el costo real.

| Nodo | h(n) | Razonamiento |
|------|------|--------------|
| A | 4 | Lejos del objetivo |
| B | 2 | A mitad de camino |
| C | 1 | Muy cerca del objetivo |
| D | 0 | Es el objetivo |

> En problemas geográficos reales se usaría distancia euclídea
> o Manhattan. Aquí usamos valores precalculados por simplicidad.

In [12]:
# Paso 2: Función heurística
# Estima el costo restante desde un nodo hasta el objetivo 'D'.
# h('D') = 0 porque ya estamos en el objetivo.

def heuristica(nodo, objetivo):
    heuristicas = {
        'A': 4,  # estimación desde A hasta D
        'B': 2,  # estimación desde B hasta D
        'C': 1,  # estimación desde C hasta D
        'D': 0   # ya estamos en el objetivo
    }
    return heuristicas.get(nodo, 0)

# Verificación
print("Valores heurísticos hacia 'D':")
for nodo in ['A', 'B', 'C', 'D']:
    print(f"  h('{nodo}') = {heuristica(nodo, 'D')}")


Valores heurísticos hacia 'D':
  h('A') = 4
  h('B') = 2
  h('C') = 1
  h('D') = 0


### Paso 3 — Implementación del algoritmo A*

El algoritmo mantiene tres estructuras de datos:

| Estructura | Tipo | Propósito |
|------------|------|-----------|
| `cola` | heap (min-priority queue) | Expandir siempre el nodo con menor f(n) |
| `costos` | dict | Guardar el menor g(n) conocido para cada nodo |
| `padres` | dict | Reconstruir el camino al final haciendo backtracking |

El flujo es:
1. Extraer el nodo con menor f(n) de la cola
2. Si es el objetivo → terminar
3. Si no → explorar sus vecinos y actualizar costos si encontramos camino más barato
4. Al terminar → reconstruir el camino siguiendo los padres desde el objetivo

In [13]:
# Paso 3: Implementación de A*

def a_estrella(grafo, inicio, objetivo):

    # Cola de prioridad: (f_score, nodo)
    # Inicializamos con el nodo de inicio
    # f = g + h → g=0 porque aún no nos hemos movido
    cola = [(0 + heuristica(inicio, objetivo), inicio)]

    # g(n): costo real acumulado para llegar a cada nodo
    # El inicio tiene costo 0
    costos = {inicio: 0}

    # Para reconstruir el camino al final
    # El inicio no tiene padre
    padres = {inicio: None}

    while cola:
        # Extraer el nodo con menor f(n)
        _, actual = heapq.heappop(cola)

        # ¿Llegamos al objetivo? → terminamos
        if actual == objetivo:
            break

        # Explorar vecinos del nodo actual
        for vecino, costo_arista in grafo[actual].items():

            # Costo real acumulado hasta este vecino
            nuevo_costo = costos[actual] + costo_arista

            # Solo actualizamos si encontramos un camino más barato
            if vecino not in costos or nuevo_costo < costos[vecino]:
                costos[vecino] = nuevo_costo
                prioridad = nuevo_costo + heuristica(vecino, objetivo)
                heapq.heappush(cola, (prioridad, vecino))
                padres[vecino] = actual  # Registramos quién es su padre

    # Reconstrucción del camino óptimo
    # Seguimos los padres desde el objetivo hasta el inicio
    camino = []
    nodo = objetivo
    while nodo is not None:
        camino.append(nodo)
        nodo = padres[nodo]
    camino.reverse()  # Invertimos: teníamos objetivo→inicio, queremos inicio→objetivo

    return camino, costos[objetivo]

print("✓ Función a_estrella definida.")

✓ Función a_estrella definida.


### Paso 4 — Ejecución y resultados

Ejecutamos A* desde la estación 'A' hasta la estación 'D'
y verificamos el camino óptimo encontrado.

In [16]:
# Paso 4: Ejecutar A*

camino_optimo, costo_total = a_estrella(grafo, 'A', 'D')

print("Camino óptimo:", " → ".join(camino_optimo))
print("Costo total:  ", costo_total)


Camino óptimo: A → B → C → D
Costo total:   4


## Actividad 3 — Optimización del Motor de Inferencia

### El problema: reglas redundantes

En el motor de la Actividad 1 cada regla era independiente.
Pero en sistemas reales las reglas pueden solaparse:

- **R1:** `temperatura > 80 AND vibración > 1` → `Sobrecarga`
- **R2:** `temperatura > 80` → `Sobrecalentamiento`
- **R3:** `vibración > 1` → `Vibración alta`

Si `t=85` y `v=1.2` las tres se activan. Sin control el motor
puede emitir diagnósticos duplicados o redundantes.

### La solución: motor data-driven

En lugar de funciones dispersas, definimos las reglas como **datos**
(lista de diccionarios). Ventajas:

- Agregar una regla = agregar un elemento a la lista
- El motor es genérico y no necesita modificarse
- Es fácil ordenar, filtrar o priorizar reglas

> Es el mismo patrón que un array de validators en un framework
> backend: la lógica de orquestación está separada de las reglas
> de negocio.

### Paso 1 — Definir las reglas como datos

Cada regla es un diccionario con cuatro campos:
- `id` — identificador para trazabilidad
- `descripcion` — texto legible de la regla
- `condicion` — función lambda que evalúa la condición
- `diagnostico` — string que se emite si la condición se cumple

Las lambdas reciben siempre `(t, v)` → temperatura y vibración.

In [19]:
# Paso 1: Reglas definidas como datos (lista de diccionarios)

reglas = [
    {
        "id"          : "R1",
        "descripcion" : "Sobrecarga (temperatura alta Y vibración alta)",
        "condicion"   : lambda t, v: t > 80 and v > 1,
        "diagnostico" : "Sobrecarga"
    },
    {
        "id"          : "R2",
        "descripcion" : "Sobrecalentamiento (solo temperatura)",
        "condicion"   : lambda t, v: t > 80,
        "diagnostico" : "Sobrecalentamiento"
    },
    {
        "id"          : "R3",
        "descripcion" : "Vibración alta (solo vibración)",
        "condicion"   : lambda t, v: v > 1,
        "diagnostico" : "Vibración alta"
    },
]

print(f"✓ {len(reglas)} reglas cargadas:")
for r in reglas:
    print(f"  [{r['id']}] {r['descripcion']}")


✓ 3 reglas cargadas:
  [R1] Sobrecarga (temperatura alta Y vibración alta)
  [R2] Sobrecalentamiento (solo temperatura)
  [R3] Vibración alta (solo vibración)


In [20]:
# Paso 2: Motor de inferencia optimizado y data-driven

def motor_inferencia_optimizado(temperatura, vibracion, reglas):
    diagnosticos = []    # Lista ordenada de diagnósticos activos
    vistos       = set() # Control de duplicados en O(1)

    for regla in reglas:
        if regla["condicion"](temperatura, vibracion):
            diag = regla["diagnostico"]
            if diag not in vistos:      # Evitar duplicados
                diagnosticos.append(diag)
                vistos.add(diag)
            print(f"  [{regla['id']}] ACTIVA  → {diag}")
        else:
            print(f"  [{regla['id']}] no activa")

    return diagnosticos

print("✓ Motor optimizado definido.")

✓ Motor optimizado definido.


### Paso 3 — Pruebas con dos casos

Probamos el motor con dos escenarios distintos:
- **Caso 1:** motor con fallo múltiple
- **Caso 2:** motor operando normalmente

In [22]:
# Paso 3: Pruebas del motor optimizado

# Caso 1: fallo múltiple
print("=" * 45)
print("CASO 1: temperatura=85, vibración=1.2")
print("=" * 45)
resultado1 = motor_inferencia_optimizado(85, 1.2, reglas)
print("Diagnósticos:", resultado1)

print()

# Caso 2: operación normal
print("=" * 45)
print("CASO 2: temperatura=70, vibración=0.5")
print("=" * 45)
resultado2 = motor_inferencia_optimizado(70, 0.5, reglas)
print("Diagnósticos:", resultado2 if resultado2 else ["Sistema operando normalmente"])


CASO 1: temperatura=85, vibración=1.2
  [R1] ACTIVA  → Sobrecarga
  [R2] ACTIVA  → Sobrecalentamiento
  [R3] ACTIVA  → Vibración alta
Diagnósticos: ['Sobrecarga', 'Sobrecalentamiento', 'Vibración alta']

CASO 2: temperatura=70, vibración=0.5
  [R1] no activa
  [R2] no activa
  [R3] no activa
Diagnósticos: ['Sistema operando normalmente']
